## 🧑‍💻 THÀNH VIÊN 2: TEXT PREPROCESSOR - NLP tiền xử lý tiếng Việt
Nhiệm vụ: Chuẩn hóa nội dung văn bản tiếng Việt
- Viết các hàm chuẩn hóa: Lowercase, xóa URLs, email, HTML tags.
- Xử lý dấu câu, emoji, ký tự đặc biệt không cần thiết.
- Xử lý Teencode, từ viết tắt tiếng Việt (vd: "ko" -> "không", "dc" -> "được").
- Loại bỏ Stopwords tiếng Việt (nếu cần thiết cho EDA).
- Lưu kết quả cuối cùng ra 3 file `_nlp_clean.csv`.


In [8]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')


## PHẦN 1: TẢI DỮ LIỆU ĐÃ CLEAN TỪ DATA CLEANER
Đọc 3 file từ Data Cleaner: `train_structural_clean.csv`, `dev_structural_clean.csv`, `test_structural_clean.csv`


In [9]:
# Đường dẫn đến dữ liệu từ Data Cleaner
DATA_CLEANER_DIR = '../Data cleaner'

datasets = {}
for name in ['train', 'dev', 'test']:
    filepath = os.path.join(DATA_CLEANER_DIR, f'{name}_structural_clean.csv')
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        datasets[name] = df
        print(f"✅ {name}_structural_clean.csv: {df.shape[0]} dòng, {df.shape[1]} cột")
        print(f"   Sample: {df['free_text'].iloc[0][:80]}...")
        print()
    else:
        print(f"❌ Không tìm thấy file {filepath}")


✅ train_structural_clean.csv: 22690 dòng, 2 cột
   Sample: Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cute coi mấy giờ này qu...

✅ dev_structural_clean.csv: 2651 dòng, 2 cột
   Sample: Coi cười xỉu...

✅ test_structural_clean.csv: 6582 dòng, 2 cột
   Sample: Đừng cố biện minh =)))) choi lon...



## PHẦN 2: XÁC ĐỊNH VIETNAMESE TEENCODE DICTIONARY & STOPWORDS
Danh sách các từ viết tắt và teencode tiếng Việt phổ biến


In [10]:
TEENCODE_DICT = {
    # Viết tắt phổ biến
    "ko": "không",
    "k": "không",
    "kh": "không",
    "khg": "không",
    "kp": "không phải",
    "kq": "không quan",
    "dc": "được",
    "đc": "được",
    "dk": "được",
    "đk": "được",
    "nc": "nước",
    "ng": "người",
    "ns": "nói",
    "mk": "mình",
    "mn": "mọi người",
    "mng": "mọi người",
    "bn": "bạn",
    "b": "bạn",
    "bro": "bạn",
    "ib": "nhắn tin",
    "rep": "trả lời",
    "vs": "với",
    "v": "với",
    "voi": "với",
    "r": "rồi",
    "rui": "rồi",
    "rii": "rồi",
    "ntn": "như thế nào",
    "j": "gì",
    "ji": "gì",
    "z": "gì",
    "gi": "gì",
    "a": "anh",
    "e": "em",
    "c": "chị",
    "đi": "đi",
    "qua": "qua",
    "trc": "trước",
    "tg": "thời gian",
    "bt": "bình thường",
    "bth": "bình thường",
    "vl": "vãi",
    "vkl": "vãi",
    "nch": "nói chuyện",
    "nt": "nhắn tin",
    "hk": "không",
    "hem": "không",
    "bi": "bị",
    "bik": "biết",
    "ck": "chồng",
    "vk": "vợ",
    "tks": "thanks",
    "thanks": "cảm ơn",
    "thks": "cảm ơn",
    "ok": "được",
    "okie": "được",
    "oke": "được",
    "plz": "làm ơn",
    "pls": "làm ơn",
    "sr": "xin lỗi",
    "sorry": "xin lỗi",
    "lun": "luôn",
    "lm": "làm",
    "đag": "đang",
    "dg": "đang",
    "trg": "trong",
    "trog": "trong",
    "cx": "cũng",
    "cg": "cũng",
    "đb": "đặc biệt",
    "cb": "chuẩn bị",
    "h": "giờ",
    "hm": "hôm",
    "dt": "điện thoại",
    "sdt": "số điện thoại",
    "fb": "facebook",
    "yt": "youtube",
    "ad": "admin",
    "mod": "moderator",
    "nx": "nhận xét",
    "đt": "điện thoại",
    "gato": "ghen ăn tức ở",
    "wtf": "what the f",
    "dm": "đ mẹ",
    "vcl": "vãi",
    "clgt": "chắc luôn",
    "oy": "rồi",
    "ùi": "rồi",
    "biet": "biết",
    "hiu": "hiểu",
    "thik": "thích",
    "hjhj": "hihi",
    "tui": "tôi",
    "mik": "mình",
    "ngta": "người ta",
    "nyc": "người yêu cũ",
    "ny": "người yêu",
    "gf": "bạn gái",
    "bf": "bạn trai",
    "sg": "sài gòn",
    "hn": "hà nội",
    "vn": "việt nam",
    "nhma": "nhưng mà",
    "nma": "nhưng mà",
    "tl": "trả lời",
    "cmn": "con mẹ nó",
}

print(f"✅ Teencode dictionary: {len(TEENCODE_DICT)} mục")


✅ Teencode dictionary: 106 mục


## PHẦN 3: ĐỊNH NGHĨA CÁC HÀM XỬ LÝ VĂN BẢN (TEXT PREPROCESSING FUNCTIONS)
Các hàm chuẩn hóa văn bản: lowercase, xóa URL, email, HTML, xử lý emoji, dấu câu...


In [11]:
# ============================================================================
# PHẦN 3: CÁC HÀM XỬ LÝ VĂN BẢN 
# ============================================================================

def remove_urls(text):
    """Xóa URL khỏi văn bản"""
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return text


def remove_emails(text):
    """Xóa email khỏi văn bản"""
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    return text


def remove_phone_numbers(text):
    """Xóa số điện thoại khỏi văn bản"""
    text = re.sub(r"(\+84|0)\d{9,10}", " ", text)
    return text


def remove_html_tags(text):
    """Xóa HTML tags"""
    text = re.sub(r"<[^>]+>", " ", text)
    return text


def remove_emojis(text):
    """Xóa emoji và các ký tự unicode đặc biệt"""
    emoji_pattern = re.compile(
        "["
        "\U0001f600-\U0001f64f"  # emoticons
        "\U0001f300-\U0001f5ff"  # symbols & pictographs
        "\U0001f680-\U0001f6ff"  # transport & map symbols
        "\U0001f1e0-\U0001f1ff"  # flags
        "\U00002702-\U000027b0"
        "\U000024c2-\U0001f251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2b55"
        "\u200d"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f"
        "\u3030"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub(" ", text)


def normalize_repeated_chars(text):
    """
    Chuẩn hóa ký tự lặp lại quá nhiều
    Ví dụ: "đẹpppppp" → "đẽpp", "hahahahaha" → "haha"
    """
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return text


def replace_teencode(text, teencode_dict=TEENCODE_DICT):
    """
    Thay thế teencode bằng từ chuẩn
    Sử dụng word boundary để tránh thay thế sai
    """
    words = text.split()
    result = []
    for word in words:
        lower_word = word.lower()
        if lower_word in teencode_dict:
            result.append(teencode_dict[lower_word])
        else:
            result.append(word)
    return " ".join(result)


def remove_special_characters(text):
    """
    Xóa ký tự đặc biệt, giữ lại chữ cái tiếng Việt, số và khoảng trắng
    """
    # Giữ lại chữ cái (bao gồm tiếng Việt), số, khoảng trắng
    text = re.sub(
        r"[^\w\sàáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    return text


def normalize_whitespace(text):
    """Chuẩn hóa khoảng trắng"""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_text(text):
    """
    Pipeline tiền xử lý hoàn chỉnh cho 1 đoạn văn bản tiếng Việt.
    Thứ tự xử lý:
    1. Kiểm tra null/NaN
    2. Chuyển lowercase
    3. Xóa HTML tags
    4. Xóa URL
    5. Xóa email
    6. Xóa số điện thoại
    7. Xóa emoji
    8. Chuẩn hóa ký tự lặp
    9. Thay thế teencode
    10. Xóa ký tự đặc biệt
    11. Chuẩn hóa khoảng trắng
    """
    # Bước 0: Kiểm tra null
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return ""

    # Bước 1: Chuyển lowercase
    text = text.lower()

    # Bước 2: Xóa HTML tags
    text = remove_html_tags(text)

    # Bước 3: Xóa URL
    text = remove_urls(text)

    # Bước 4: Xóa email
    text = remove_emails(text)

    # Bước 5: Xóa số điện thoại
    text = remove_phone_numbers(text)

    # Bước 6: Xóa emoji
    text = remove_emojis(text)

    # Bước 7: Chuẩn hóa ký tự lặp
    text = normalize_repeated_chars(text)

    # Bước 8: Thay thế teencode
    text = replace_teencode(text)

    # Bước 9: Xóa ký tự đặc biệt
    text = remove_special_characters(text)

    # Bước 10: Chuẩn hóa khoảng trắng
    text = normalize_whitespace(text)

    return text

print("✅ Các hàm xử lý văn bản đã được định nghĩa")


✅ Các hàm xử lý văn bản đã được định nghĩa


## PHẦN 4: KIỂM TRA CÁC HÀM XỬ LÝ VĂN BẢN
Test các hàm preprocessing trên một số mẫu dữ liệu


In [ ]:
# Test các hàm trên một số mẫu dữ liệu
test_samples = [
    "Ko biet j het, mn rep ntn vậy? qá trời luôn :v", 
    "Hnay mik vs bn đi cf, kq dc j mà còn bị la nữa", 
    "Inbox ib sdt 0912345678 or email test@example.com nhé", 
    "Vkl, thik thì okie, hem thì thui kkk!!! @#$%^&*()", 
    "Tui đag trg lớp, nch vs ngta xong rui sẽ tl sau nhe", 
]

print("=" * 80)
print("🧪 TEST PREPROCESSING FUNCTIONS")
print("=" * 80)

for i, sample in enumerate(test_samples, 1):
    print(f"\n📝 SAMPLE {i}:")
    print(f"   ⚪ Input:  {sample}")
    processed = preprocess_text(sample)
    print(f"   🟢 Output: {processed}")

print("\n" + "=" * 80)


🧪 TEST PREPROCESSING FUNCTIONS

📝 SAMPLE 1:
   ⚪ Input:  Chi ba vang ngoc dep va tre mai 😂😂
   🟢 Output: chi ba vang ngoc dep va tre mai

📝 SAMPLE 2:
   ⚪ Input:  Ko dc làm gì thì k làm vs được thôi :v lol lol lol
   🟢 Output: không được làm gì thì không làm với được thôi v lol lol lol

📝 SAMPLE 3:
   ⚪ Input:  Đi link http://example.com hoặc gửi email: test@example.com
   🟢 Output: đi link hoặc gửi email

📝 SAMPLE 4:
   ⚪ Input:  Cực kì thuyết phục!!! @#$%^&*() hahaha
   🟢 Output: cực kì thuyết phục hahaha

📝 SAMPLE 5:
   ⚪ Input:  Lươi thìng con trai ko nên dùng Facebook 💔💔💔
   🟢 Output: lươi thìng con trai không nên dùng facebook



## PHẦN 4: CHẠY PIPELINE NLP PREPROCESSING TRÊN TẤT CẢ DỮ LIỆU
Áp dụng các hàm tiền xử lý NLP cho cả 3 tập dữ liệu đã clean từ mem1

**Lưu ý**: Các bước xử lý missing values và duplicates đã hoàn tất ở mem1 (Data Cleaner),
nên ở đây chỉ cần xóa dòng rỗng sau khi NLP preprocessing


In [13]:
print("=" * 80)
print("🚀 BẮT ĐẦU NLP PREPROCESSING CHO CẢ 3 TẬP DỮ LIỆU")
print("=" * 80)

datasets_nlp_clean = {}
OUTPUT_DIR = '.'  # Lưu vào thư mục hiện tại

for name in ['train', 'dev', 'test']:
    if name not in datasets:
        print(f"❌ {name} dataset not found!")
        continue
    
    df = datasets[name].copy()
    print(f"\n⏳ Đang xử lý tập: {name.upper()}")
    print(f"   📊 Tổng dòng (đã clean từ mem1): {len(df)}")
    
    # Áp dụng NLP preprocessing cho từng dòng
    df['free_text_clean'] = df['free_text'].apply(preprocess_text)
    
    # Kiểm tra số dòng rỗng sau NLP preprocessing
    empty_count = (df['free_text_clean'] == '').sum()
    print(f"   ⚠️  Số dòng rỗng sau NLP: {empty_count}")
    
    # Xóa các dòng rỗng (chỉ xóa từ NLP preprocessing, không xóa null/duplicates vì đã làm ở mem1)
    if empty_count > 0:
        df = df[df['free_text_clean'] != '']
        print(f"   ✅ Đã xóa {empty_count} dòng rỗng. Còn lại: {len(df)} dòng")
    
    # Lưu vào dictionary
    datasets_nlp_clean[name] = df
    
    # Lưu file ra CSV (ghi đè lên file từ mem1 với cột clean)
    output_filename = f'{name}_clean.csv'
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   💾 Đã lưu file: {output_path}")
    
    # Thống kê sau NLP
    print(f"\n   📈 THỐNG KÊ SAU NLP:\"")
    print(f"      Độ dài text trung bình: {df['free_text_clean'].str.len().mean():.1f} ký tự")
    print(f"      Số từ trung bình: {df['free_text_clean'].str.split().str.len().mean():.1f} từ")
    print(f"      Phân bố nhãn: {df['label_id'].value_counts().sort_index().to_dict()}")
    
    # In một vài mẫu để kiểm tra
    print(f"\n   📋 Mẫu dữ liệu - Trước và Sau NLP:\"")
    for idx in range(min(2, len(df))):
        print(f"      Before: {df['free_text'].iloc[idx][:60]}...")
        print(f"      After:  {df['free_text_clean'].iloc[idx][:60]}...")

print("\n" + "=" * 80)
print("✅ TỔNG KẾT NLP PREPROCESSING")
print("=" * 80)

for name in ['train', 'dev', 'test']:
    if name in datasets_nlp_clean:
        df = datasets_nlp_clean[name]
        print(f"📊 {name}: {len(df)} dòng")

print(f"\n🎯 HOÀN TẤT NHIỆM VỤ THÀNH VIÊN 2: TEXT PREPROCESSOR (NLP)")
print("✅ Dữ liệu đã sẵn sàng cho Thành viên 3 (EDA)")


🚀 BẮT ĐẦU NLP PREPROCESSING CHO CẢ 3 TẬP DỮ LIỆU

⏳ Đang xử lý tập: TRAIN
   📊 Tổng dòng (đã clean từ mem1): 22690
   ⚠️  Số dòng rỗng sau NLP: 180
   ✅ Đã xóa 180 dòng rỗng. Còn lại: 22510 dòng
   💾 Đã lưu file: .\train_clean.csv

   📈 THỐNG KÊ SAU NLP:"
      Độ dài text trung bình: 48.1 ký tự
      Số từ trung bình: 11.7 từ
      Phân bố nhãn: {0: 18473, 1: 1542, 2: 2495}

   📋 Mẫu dữ liệu - Trước và Sau NLP:"
      Before: Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cut...
      After:  em được làm fan cứng luôn rồi nè reaction quá hay quá cute c...
      Before: Đúng là bọn mắt híp lò xo thụt :))) bên việt nam t cái này r...
      After:  đúng là bọn mắt híp lò xo thụt bên việt nam t cái này ra các...

⏳ Đang xử lý tập: DEV
   📊 Tổng dòng (đã clean từ mem1): 2651
   ⚠️  Số dòng rỗng sau NLP: 18
   ✅ Đã xóa 18 dòng rỗng. Còn lại: 2633 dòng
   💾 Đã lưu file: .\dev_clean.csv

   📈 THỐNG KÊ SAU NLP:"
      Độ dài text trung bình: 47.1 ký tự
      Số từ trung bình: 11.5 từ